In [ ]:
import psycopg2
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from datetime import datetime

conn = psycopg2.connect(
    host="localhost",
    database="ecommerce_rec",
    user="postgres",
    password="votre_mot_de_passe",
    port="5432"
)
cursor = conn.cursor()

# DONNÉES
cursor.execute("SELECT IDUser, IDProduit, Note, DateNote FROM Notes")
notes = cursor.fetchall()
cursor.execute("SELECT IDUser, NomUser FROM Users ORDER BY IDUser")
users = cursor.fetchall()
user_ids = [u[0] for u in users]
user_names = {u[0]: u[1] for u in users}
cursor.execute("SELECT ID, NomPdt FROM Produit ORDER BY ID")
produits = cursor.fetchall()
produit_ids = [p[0] for p in produits]
produit_names = {p[0]: p[1] for p in produits}

u_idx = {u: i for i, u in enumerate(user_ids)}
p_idx = {p: i for i, p in enumerate(produit_ids)}

# MATRICE
matrice = np.zeros((len(user_ids), len(produit_ids)))
for u, p, n, d in notes:
    matrice[u_idx[u]][p_idx[p]] = n
print("✅ Matrice Users x Produits créée")

# SIMILARITÉ USERS
sim_users = cosine_similarity(matrice)
print("✅ Similarité entre users calculée")
print(np.round(sim_users, 2))

# PRÉDIRE NOTE
def predire_note(id_user, id_produit, K=2):
    u = u_idx[id_user]
    p = p_idx.get(id_produit)
    if p is None: return 0
    sims = [(sim_users[u][v], v) for v in range(len(user_ids))
            if v != u and matrice[v][p] > 0]
    sims.sort(reverse=True)
    top_k = sims[:K]
    if not top_k: return 0
    num = sum(s * matrice[v][p] for s, v in top_k)
    den = sum(abs(s) for s, v in top_k)
    return round(num / den, 2) if den > 0 else 0

# TOP-N COLLABORATIF
def recommander_collaboratif(id_user, N=3):
    u = u_idx[id_user]
    scores = []
    for id_p in produit_ids:
        p = p_idx[id_p]
        if matrice[u][p] == 0:
            note_pred = predire_note(id_user, id_p)
            scores.append((id_p, note_pred))
    scores.sort(key=lambda x: x[1], reverse=True)
    top = scores[:N]
    print(f"\n👤 Top {N} pour {user_names[id_user]}:")
    for id_p, score in top:
        print(f"  → {produit_names[id_p]} | Note prédite: {score}")
        print(f"     💡 Recommandé car des users similaires ont aimé ce produit")
    return top

# TIME-AWARE
def recommander_recent(id_user, N=3):
    aujourd_hui = datetime.now()
    scores = {}
    for u, p, n, d in notes:
        if u == id_user: continue
        jours = (aujourd_hui - datetime(d.year, d.month, d.day)).days
        poids = 1 / (1 + jours / 30)
        scores[p] = scores.get(p, 0) + n * poids
    u = u_idx[id_user]
    resultats = [(id_p, s) for id_p, s in scores.items()
                 if matrice[u][p_idx[id_p]] == 0]
    resultats.sort(key=lambda x: x[1], reverse=True)
    top = resultats[:N]
    print(f"\n⏰ Top {N} récents pour {user_names[id_user]}:")
    for id_p, score in top:
        print(f"  → {produit_names[id_p]} | Score temporel: {round(score,2)}")
        print(f"     💡 Recommandé car tendance récente")
    return top

# HYBRIDE
def recommander_hybride(id_user, id_produit_prefere, N=3):
    r_collab = recommander_collaboratif(id_user, N)
    r_recent = recommander_recent(id_user, N)
    cursor.execute("""
        SELECT IDProduit2, Score FROM SimilariteContenu
        WHERE IDProduit1=%s ORDER BY Score DESC LIMIT %s
    """, (id_produit_prefere, N))
    r_contenu = cursor.fetchall()
    scores = {}
    for id_p, s in r_collab:
        scores[id_p] = scores.get(id_p, 0) + 0.4 * s
    for id_p, s in r_recent:
        scores[id_p] = scores.get(id_p, 0) + 0.2 * s
    for id_p, s in r_contenu:
        scores[id_p] = scores.get(id_p, 0) + 0.4 * s
    resultats = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:N]
    print(f"\n🔀 Hybride pour {user_names[id_user]}:")
    for id_p, score in resultats:
        print(f"  → {produit_names[id_p]} | Score: {round(score,2)}")
        print(f"     💡 Recommandé par combinaison contenu+collaboratif+temps")
    return resultats

# TESTS
recommander_collaboratif(1, 3)
recommander_collaboratif(5, 3)
recommander_recent(1, 3)
recommander_hybride(2, 1, 3)

conn.close()
print("\n✅ Filtrage collaboratif + hybride terminé!")